Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("why do parrots speak?")
response.content

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    # Note: Typically tools query a real external API. Calling model.invoke
    # within the tool itself is a placeholder implementation.
    response = model.invoke(f"what is the current weather in {location}")
    return response.content

model_with_tool = model.bind_tools([get_weather])

response = model_with_tool.invoke("What is the weather in Tokyo?")
response.tool_calls

### How to Get the Final Answer

To get the final answer when a model requests a tool call, you must:
1. **Execute the tool** with the arguments specified in `response.tool_calls`.
2. **Construct a `ToolMessage`** containing the results and the `tool_call_id`.
3. **Send the conversation history** (user prompt, model's tool request, and tool message) back to the model.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1. Create conversation history starting with the user query
messages = [HumanMessage(content="What is the weather in Tokyo?")]

# 2. Call the model with tool binding
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)  # Add model's tool call message to history

# 3. Execute the requested tool(s)
for tool_call in ai_msg.tool_calls:
    # Map the tool name to the actual tool function/object
    tool_map = {"get_weather": get_weather}
    selected_tool = tool_map[tool_call["name"]]
    
    # Run the tool
    tool_output = selected_tool.invoke(tool_call["args"])
    
    # Append the result as a ToolMessage linked by tool_call_id
    messages.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"]))

# 4. Get the final response from the model
final_response = model_with_tool.invoke(messages)
print(final_response.content)

### Option 2: Using an Agent to Run the Tool Automatically

If you want LangChain to automatically execute the tool and return the final answer, you can wrap the model and tool in an **Agent** using `create_tool_calling_agent` and `AgentExecutor`.

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# 1. Define a prompt template for the agent
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 2. Create the tool-calling agent
agent = create_tool_calling_agent(model, [get_weather], prompt)

# 3. Combine agent and tools into an AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=[get_weather], verbose=True)

# 4. Invoke the agent to run the entire loop automatically
result = agent_executor.invoke({"input": "What is the weather in Tokyo?"})
print(result["output"])